# P9 回帰の関門（CI ゲート）

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

費用を下げるために分類の依頼文を短くした「候補」を、いまの本番（基準）と同じ問いで比べ、出してよいかを非劣性・層の床・無効な出力の 3 つの決まりで決めます。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 1. 基準と候補を決める
関門の決まり（gate_config.json）と、短くした依頼文。

In [ ]:
from common import kit, stats
show_file("lab09_ci_gate/gate_config.json")
full = open("lab03_classify_extract/prompts/classify.md", encoding="utf-8").read()
full = full.format(guide=kit.load_text("guide_inquiries.md"))
short = open("lab09_ci_gate/prompts/classify_trimmed.md", encoding="utf-8").read()
print(f"\n基準の依頼文（分類の手引きつき）{len(full)} 字 / 候補の依頼文（短くした）{len(short)} 字")
show_file("lab09_ci_gate/prompts/classify_trimmed.md", 2, 4)

## 2. 候補を v2 難問 48 問で回す
講師の本物の実行（Sonnet と Opus）の記録を再生します。

In [ ]:
import io, contextlib
P = "replay" if PROVIDER == "mock" else PROVIDER
rc = lab("run_candidate", "lab09_ci_gate")
with safe():
    for m in ("claude-sonnet-5-5", "claude-opus-5-5"):
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            rc.main(["--provider", P, "--all", "--model", m])
        print(f"{m:<18}", [l for l in buf.getvalue().splitlines() if "候補の正答" in l][0])
print("基準（全文の手引き）は両モデルとも 48/48（lab03）")

## 3. 非劣性 ―「勝った」ではなく「負けていない」を言う
差の下限（片側 95%）が −0.03 より上なら「負けていない」。

In [ ]:
gate = lab("gate", "lab09_ci_gate")
cfg = kit.read_json("lab09_ci_gate/gate_config.json")
pairs, _ = gate.default_pairs()
for name, bp, cp in pairs:
    r = gate.evaluate(gate.load(bp), gate.load(cp), cfg)
    ok = r["lower"] > -cfg["margin"]
    print(f"■ {name}")
    print(f"   差 {r['diff']:+.3f} / 下限 {r['lower']:+.3f} / 線 −{cfg['margin']:.2f}"
          f" → {'負けていない（通す）' if ok else '負けていないとは言えない（止める）'}")

## 4. 層の床 ― どの種類も大きくは落とさない
全体の差だけでなく、種類（scenario）ごとに 25 ポイント超の落ち込みを止めます。

In [ ]:
show(gate.evaluate, 15, 25)
r = gate.evaluate(gate.load(pairs[0][1]), gate.load(pairs[0][2]), cfg)
print(f"\n{pairs[0][0]}:")
for s, k, ab, cb in r["slices"]:
    print(f"   {s:14s} n={k}  基準 {ab:.0%} → 候補 {cb:.0%}  {'✗ 床を割った' if ab - cb > cfg['floor'] else ''}")

## 5. gate.py を回す ― 終了コードで CI を止める
3 つの組をまとめて判定します（長い行は後ろを省いて表示）。

In [ ]:
def cut(s, w=100):                  # 表示幅（全角 2）で切って、後ろを「…」にする
    s, n = s.replace("\n", " "), 0
    for i, ch in enumerate(s):
        n += 2 if ord(ch) > 0x2E80 else 1
        if n > w:
            return s[:i] + "…"
    return s
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    code = gate.main([])
for line in buf.getvalue().splitlines()[2:]:
    if line.strip():
        print(cut(line, 104))
print("gate.py の終了コード:", code, "（1 = 止める。CI ならここで止まる）")

## 6. CI に入れる形
記録の再生だけで関門を回す例（このキットでは有効にしていない）。

In [ ]:
print("依頼文・モデルを変えた PR で、記録の再生だけで関門を回す（新しい依頼は API を呼ばずに止まる）")
show_file("lab09_ci_gate/ci_example.yml", 3)

## 7. 評価の道具の失敗 ― 予算切れを「無効」と数えない
予算の上限・記録なしで止まった呼び出しは、モデルの無効な出力ではない。

In [ ]:
from common import llm as L
show_file("lab09_ci_gate/run_candidate.py", 59, 64)
print()
show(L._anthropic, 0, 2)

## 8. 例題 ― 満点同士でも差の下限は −2.2 ポイント
食い違い 0 件のときは、Wilson の上限（片側 95% = 両側 90%）で幅を付けます。

In [ ]:
n = 120
z = stats.z(0.90)
up = z * z / (n + z * z)           # 0/n のときの Wilson の上限は z²/(n+z²) にまとまる
print(f"z = {z:.4f}、上限 = {z * z:.4f} / ({n} + {z * z:.4f}) = {up:.4f}")
print(f"stats.wilson(0, 120, conf=0.90) の上限 = {stats.wilson(0, n, conf=0.90)[1]:.4f}")
r = gate.evaluate(gate.load(pairs[2][1]), gate.load(pairs[2][2]), cfg)
print(f"関門の計算（Opus → Sonnet）: 下限 {r['lower']:+.3f} > −0.03 → 通す、費用 {r['cost_base']:.2f} → {r['cost_cand']:.2f} USD")
for n in (40, 80, 100, 300):
    lo = -stats.wilson(0, n, conf=0.90)[1]
    print(f"   {n:3d} 問・満点同士なら: 下限 {100 * lo:+.1f} ポイント → {'通す' if lo > -0.03 else '止める'}")

## 9. 例題 ― 依頼文を短くした候補を止める
関門の判定を、数字を追って確かめます（Sonnet・v2 難問 48 問）。

In [ ]:
r = gate.evaluate(gate.load(pairs[0][1]), gate.load(pairs[0][2]), cfg)
nb_, nc_ = round(r["base_acc"] * r["n"]), round(r["cand_acc"] * r["n"])
print(f"基準 {nb_}/{r['n']} → 候補 {nc_}/{r['n']}、差 = ({nc_} − {nb_}) / {r['n']} = {r['diff']:+.3f}")
print(f"片側 95% の下限 {r['lower']:+.3f} < −0.03 → 非劣性を満たさない")
for s, k, ab, cb in r["slices"]:
    if ab - cb > cfg["floor"]:
        print(f"層の床: {s} {ab:.0%} → {cb:.0%}（{100 * (ab - cb):.0f} ポイント落ちた）")
cut_ = 1 - r["cost_cand"] / r["cost_base"]
print(f"費用 {r['cost_base']:.4f} → {r['cost_cand']:.4f} USD（{100 * cut_:.0f}% 安い）。安くなった分より、壊れた分が大きい")
print("判定:", "通す" if r["pass"] else "止める")